# Gold Layer Metrics From `silver_df.csv`

This notebook builds local Gold Layer metric datasets from the prepared Silver CSV snapshot.

Input:
- `data/gold/silver_df.csv`

Output:
- `data/processed/gold/metrics/<metric_name>/schema_v=1/batch_id=<uuid>/...`

The implementation is local-only and notebook-first.

In [1]:
from pathlib import Path
import csv
import json
import uuid
from collections import defaultdict

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("local-gold-layer-metrics")
    .master("local[*]")
    .getOrCreate()
)
spark.conf.set("spark.sql.session.timeZone", "UTC")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Gold Layer":
    PROJECT_ROOT = PROJECT_ROOT.parent

SILVER_CSV_PATH = PROJECT_ROOT / "data" / "gold" / "silver_df.csv"
GOLD_ROOT = PROJECT_ROOT / "data" / "processed" / "gold" / "metrics"
AUTHOR_SURNAME = "Yermolovych"
SCHEMA_VERSION = 1

print("Project root:", PROJECT_ROOT)
print("Input CSV exists:", SILVER_CSV_PATH.exists(), SILVER_CSV_PATH)
print("Gold root:", GOLD_ROOT)

Project root: E:\Academy of Mohyla\Course3_Stage2\bigData
Input CSV exists: True E:\Academy of Mohyla\Course3_Stage2\bigData\data\gold\silver_df.csv
Gold root: E:\Academy of Mohyla\Course3_Stage2\bigData\data\processed\gold\metrics


## 1. Read Silver CSV

In [2]:
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(SILVER_CSV_PATH))
)

drop_candidates = [name for name in raw_df.columns if name is None or name == "" or name.startswith("_c") or name == "Unnamed: 0"]
silver_df = raw_df.drop(*drop_candidates) if drop_candidates else raw_df

silver_df.printSchema()
silver_df.show(5, truncate=False)
print("Dropped synthetic columns:", drop_candidates)

root
 |-- vehicle_id: integer (nullable = true)
 |-- route_id: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- lng: double (nullable = true)
 |-- speed: double (nullable = true)
 |-- dir: double (nullable = true)
 |-- time_unix: integer (nullable = true)
 |-- ingest_date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- speed_gps: double (nullable = true)
 |-- dt_sec: double (nullable = true)
 |-- dist_km: double (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- period: string (nullable = true)
 |-- lat_grid: double (nullable = true)
 |-- lng_grid: double (nullable = true)
 |-- is_congested: integer (nullable = true)
 |-- is_alert: integer (nullable = true)
 |-- is_post_alert: integer (nullable = true)
 |-- min_since_alert: integer (nullable = true)
 |-- temp_c: double (nullable = true)
 |-- precip_mm: double (nullable = true)
 |-- rain_mm: double (nullable = true)
 |-- snow_cm: double (nullable = true)
 |-- snow_depth_m: double (nulla

## 2. Normalize and Clean Silver Rows

In [3]:
typed_df = (
    silver_df
    .withColumn("vehicle_id", F.col("vehicle_id").cast("long"))
    .withColumn("route_id", F.col("route_id").cast("long"))
    .withColumn("lat", F.col("lat").cast("double"))
    .withColumn("lng", F.col("lng").cast("double"))
    .withColumn("speed", F.col("speed").cast("double"))
    .withColumn("dir", F.col("dir").cast("double"))
    .withColumn("time_unix", F.col("time_unix").cast("long"))
    .withColumn("ingest_date", F.col("ingest_date").cast("string"))
    .withColumn("hour", F.col("hour").cast("int"))
    .withColumn("speed_gps", F.col("speed_gps").cast("double"))
    .withColumn("dt_sec", F.col("dt_sec").cast("double"))
    .withColumn("dist_km", F.col("dist_km").cast("double"))
    .withColumn("day_of_week", F.col("day_of_week").cast("int"))
    .withColumn("period", F.col("period").cast("string"))
    .withColumn("lat_grid", F.col("lat_grid").cast("double"))
    .withColumn("lng_grid", F.col("lng_grid").cast("double"))
    .withColumn("is_congested", F.col("is_congested").cast("int"))
    .withColumn("is_alert", F.col("is_alert").cast("int"))
    .withColumn("is_post_alert", F.col("is_post_alert").cast("int"))
    .withColumn("min_since_alert", F.col("min_since_alert").cast("double"))
    .withColumn("temp_c", F.col("temp_c").cast("double"))
    .withColumn("precip_mm", F.col("precip_mm").cast("double"))
    .withColumn("rain_mm", F.col("rain_mm").cast("double"))
    .withColumn("snow_cm", F.col("snow_cm").cast("double"))
    .withColumn("snow_depth_m", F.col("snow_depth_m").cast("double"))
    .withColumn("wind_kmh", F.col("wind_kmh").cast("double"))
    .withColumn("visibility_m", F.col("visibility_m").cast("double"))
    .withColumn("cloud_pct", F.col("cloud_pct").cast("double"))
    .withColumn("is_freezing", F.col("is_freezing").cast("int"))
    .withColumn("is_snowing", F.col("is_snowing").cast("int"))
    .withColumn("is_raining", F.col("is_raining").cast("int"))
    .withColumn("bad_visibility", F.col("bad_visibility").cast("int"))
    .withColumn("gold_author_surname", F.lit(AUTHOR_SURNAME))
    .withColumn("schema_v", F.lit(SCHEMA_VERSION))
)

valid_condition = (
    F.col("vehicle_id").isNotNull()
    & F.col("route_id").isNotNull()
    & F.col("lat").isNotNull()
    & F.col("lng").isNotNull()
    & F.col("speed").isNotNull()
    & F.col("dt_sec").isNotNull()
    & F.col("dist_km").isNotNull()
    & F.col("ingest_date").isNotNull()
    & F.col("hour").isNotNull()
    & F.col("period").isNotNull()
    & F.col("lat_grid").isNotNull()
    & F.col("lng_grid").isNotNull()
    & F.col("is_congested").isNotNull()
    & F.col("is_alert").isNotNull()
    & F.col("is_post_alert").isNotNull()
    & F.col("temp_c").isNotNull()
    & F.col("precip_mm").isNotNull()
    & F.col("rain_mm").isNotNull()
    & F.col("snow_cm").isNotNull()
    & F.col("snow_depth_m").isNotNull()
    & F.col("wind_kmh").isNotNull()
    & F.col("visibility_m").isNotNull()
    & F.col("cloud_pct").isNotNull()
    & F.col("is_freezing").isNotNull()
    & F.col("is_snowing").isNotNull()
    & F.col("is_raining").isNotNull()
    & F.col("bad_visibility").isNotNull()
    & (F.col("speed") >= 0)
    & (F.col("speed") <= 150)
    & (F.col("dt_sec") > 0)
    & (F.col("dist_km") >= 0)
    & (F.col("lat").between(-90.0, 90.0))
    & (F.col("lng").between(-180.0, 180.0))
    & (F.col("temp_c").between(-60.0, 100.0))
    & (F.col("precip_mm") >= 0)
    & (F.col("rain_mm") >= 0)
    & (F.col("snow_cm") >= 0)
    & (F.col("snow_depth_m") >= 0)
    & (F.col("wind_kmh") >= 0)
    & (F.col("visibility_m") >= 0)
    & (F.col("cloud_pct").between(0.0, 100.0))
    & (F.col("hour").between(0, 23))
)

invalid_df = typed_df.filter(~valid_condition)
clean_df = typed_df.filter(valid_condition)

rows_before = typed_df.count()
rows_after = clean_df.count()
rows_rejected = rows_before - rows_after

print("Rows before cleaning:", rows_before)
print("Rows after cleaning:", rows_after)
print("Rows rejected:", rows_rejected)
clean_df.show(5, truncate=False)

Rows before cleaning: 9009537
Rows after cleaning: 9009537
Rows rejected: 0
+----------+--------+--------+--------+---------+-----+----------+-----------+----+---------+------+-----------+-----------+------------+--------+------------------+------------+--------+-------------+---------------+------+---------+-------+-------+------------+--------+------------+---------+-----------+----------+----------+--------------+-------------------+--------+
|vehicle_id|route_id|lat     |lng     |speed    |dir  |time_unix |ingest_date|hour|speed_gps|dt_sec|dist_km    |day_of_week|period      |lat_grid|lng_grid          |is_congested|is_alert|is_post_alert|min_since_alert|temp_c|precip_mm|rain_mm|snow_cm|snow_depth_m|wind_kmh|visibility_m|cloud_pct|is_freezing|is_snowing|is_raining|bad_visibility|gold_author_surname|schema_v|
+----------+--------+--------+--------+---------+-----+----------+-----------+----+---------+------+-----------+-----------+------------+--------+------------------+-----------

## 3. Build Gold Metric DataFrames

In [4]:
traffic_activity_daily = (
    clean_df
    .groupBy("ingest_date")
    .agg(
        F.countDistinct("vehicle_id").alias("active_vehicles"),
        F.countDistinct("route_id").alias("active_routes"),
        F.count("*").alias("movement_points"),
        F.avg("speed").alias("avg_speed_kmh")
    )
)

avg_speed_by_period = (
    clean_df
    .groupBy("ingest_date", "period")
    .agg(
        F.avg("speed").alias("avg_speed_kmh"),
        F.sum("is_congested").alias("congested_points"),
        F.count("*").alias("points_cnt")
    )
)

congestion_weather_daily = (
    clean_df
    .groupBy("ingest_date")
    .agg(
        F.avg("is_congested").alias("congested_share"),
        F.avg("is_alert").alias("alert_share"),
        F.avg("is_post_alert").alias("post_alert_share"),
        F.avg("temp_c").alias("avg_temp_c"),
        F.avg("precip_mm").alias("avg_precip_mm"),
        F.avg("bad_visibility").alias("bad_visibility_share"),
        F.avg("is_freezing").alias("freezing_share")
    )
)

route_performance_daily = (
    clean_df
    .groupBy("ingest_date", "route_id")
    .agg(
        F.countDistinct("vehicle_id").alias("vehicles_cnt"),
        F.count("*").alias("points_cnt"),
        F.avg("speed").alias("avg_speed_kmh"),
        F.avg("dist_km").alias("avg_dist_km"),
        F.avg("is_congested").alias("congestion_rate")
    )
    .filter(F.col("points_cnt") > 20)
)

geo_hotspots_daily = (
    clean_df
    .groupBy("ingest_date", "lat_grid", "lng_grid")
    .agg(
        F.count("*").alias("points_cnt"),
        F.avg("speed").alias("avg_speed_kmh"),
        F.avg("is_congested").alias("congestion_rate"),
        F.avg("is_alert").alias("alerts_rate"),
        F.avg("bad_visibility").alias("bad_visibility_rate")
    )
)

print("traffic_activity_daily")
traffic_activity_daily.show(5, truncate=False)
print("avg_speed_by_period")
avg_speed_by_period.show(5, truncate=False)
print("congestion_weather_daily")
congestion_weather_daily.show(5, truncate=False)
print("route_performance_daily")
route_performance_daily.show(5, truncate=False)
print("geo_hotspots_daily")
geo_hotspots_daily.show(5, truncate=False)

traffic_activity_daily
+-----------+---------------+-------------+---------------+------------------+
|ingest_date|active_vehicles|active_routes|movement_points|avg_speed_kmh     |
+-----------+---------------+-------------+---------------+------------------+
|2026-02-27 |697            |155          |371452         |16.03857472362904 |
|2026-02-15 |378            |143          |187815         |17.2662390312899  |
|2026-02-05 |506            |153          |175831         |15.157986418219057|
|2026-02-25 |510            |147          |242863         |16.798924089339422|
|2026-03-10 |790            |158          |456724         |16.013335711795712|
+-----------+---------------+-------------+---------------+------------------+
only showing top 5 rows

avg_speed_by_period
+-----------+------------+------------------+----------------+----------+
|ingest_date|period      |avg_speed_kmh     |congested_points|points_cnt|
+-----------+------------+------------------+----------------+----------+

## 4. Local Writer Helper

Spark computes the metrics, while the local writer stores each metric dataset as partitioned CSV files.
This avoids Windows `winutils` issues and still keeps `.repartition(8)` before write.

In [5]:
written_paths = []

def write_metric_dataset(df, metric_name, partition_cols):
    batch_id = str(uuid.uuid4())
    base_dir = GOLD_ROOT / metric_name / f"schema_v={SCHEMA_VERSION}" / f"batch_id={batch_id}"
    base_dir.mkdir(parents=True, exist_ok=True)

    enriched_df = (
        df
        .withColumn("gold_author_surname", F.lit(AUTHOR_SURNAME))
        .withColumn("schema_v", F.lit(SCHEMA_VERSION))
        .withColumn("batch_id", F.lit(batch_id))
        .repartition(8)
    )

    writers = {}
    try:
        for row in enriched_df.toLocalIterator():
            record = row.asDict(recursive=True)
            partition_values = [str(record[col]) for col in partition_cols]
            partition_dir = base_dir
            for col_name, value in zip(partition_cols, partition_values):
                partition_dir = partition_dir / f"{col_name}={value}"
            partition_dir.mkdir(parents=True, exist_ok=True)
            file_path = partition_dir / f"part-{batch_id}.csv"

            if file_path not in writers:
                ordered_columns = list(record.keys())
                file_handle = file_path.open("w", newline="", encoding="utf-8")
                writer = csv.DictWriter(file_handle, fieldnames=ordered_columns)
                writer.writeheader()
                writers[file_path] = (file_handle, writer)

            writers[file_path][1].writerow(record)
    finally:
        for file_handle, _writer in writers.values():
            file_handle.close()

    written_paths.append(str(base_dir))
    print(f"WRITTEN: {base_dir}")
    return base_dir

traffic_activity_path = write_metric_dataset(traffic_activity_daily, "traffic_activity_daily", ["ingest_date"])
avg_speed_by_period_path = write_metric_dataset(avg_speed_by_period, "avg_speed_by_period", ["ingest_date"])
congestion_weather_daily_path = write_metric_dataset(congestion_weather_daily, "congestion_weather_daily", ["ingest_date"])
route_performance_daily_path = write_metric_dataset(route_performance_daily, "route_performance_daily", ["ingest_date"])
geo_hotspots_daily_path = write_metric_dataset(geo_hotspots_daily, "geo_hotspots_daily", ["ingest_date"])


WRITTEN: E:\Academy of Mohyla\Course3_Stage2\bigData\data\processed\gold\metrics\traffic_activity_daily\schema_v=1\batch_id=42104c1c-5208-4fd9-9cdb-7e3675cd951d
WRITTEN: E:\Academy of Mohyla\Course3_Stage2\bigData\data\processed\gold\metrics\avg_speed_by_period\schema_v=1\batch_id=812f0f2e-c3ef-4647-a06c-23fe5f5c288b
WRITTEN: E:\Academy of Mohyla\Course3_Stage2\bigData\data\processed\gold\metrics\congestion_weather_daily\schema_v=1\batch_id=558f2aaf-fa46-44a7-adc1-92bdc5467334
WRITTEN: E:\Academy of Mohyla\Course3_Stage2\bigData\data\processed\gold\metrics\route_performance_daily\schema_v=1\batch_id=aa3ec766-b6b6-44bc-94ef-18bd6268c4b6
WRITTEN: E:\Academy of Mohyla\Course3_Stage2\bigData\data\processed\gold\metrics\geo_hotspots_daily\schema_v=1\batch_id=2934aac3-3906-4b43-9db9-b0d509369e86


## 5. Read Back Gold Datasets As Proof

In [7]:
def load_metric_rows(metric_path, limit=5):
    files = sorted(Path(metric_path).rglob("*.csv"))
    rows = []
    for file_path in files:
        with file_path.open("r", encoding="utf-8", newline="") as handle:
            reader = csv.DictReader(handle)
            for row in reader:
                rows.append(row)
    return files, rows[:limit], len(rows)

traffic_files, traffic_rows_preview, traffic_rows_total = load_metric_rows(traffic_activity_path)
route_files, route_rows_preview, route_rows_total = load_metric_rows(route_performance_daily_path)

print("traffic_activity_daily files:", len(traffic_files))
print("traffic_activity_daily rows:", traffic_rows_total)
for row in traffic_rows_preview:
    print(row)

print("\nroute_performance_daily files:", len(route_files))
print("route_performance_daily rows:", route_rows_total)
for row in route_rows_preview:
    print(row)


traffic_activity_daily files: 36
traffic_activity_daily rows: 36
{'ingest_date': '2026-02-02', 'active_vehicles': '284', 'active_routes': '116', 'movement_points': '13298', 'avg_speed_kmh': '18.325358808722783', 'gold_author_surname': 'Yermolovych', 'schema_v': '1', 'batch_id': '42104c1c-5208-4fd9-9cdb-7e3675cd951d'}
{'ingest_date': '2026-02-03', 'active_vehicles': '524', 'active_routes': '158', 'movement_points': '148795', 'avg_speed_kmh': '16.786642555791396', 'gold_author_surname': 'Yermolovych', 'schema_v': '1', 'batch_id': '42104c1c-5208-4fd9-9cdb-7e3675cd951d'}
{'ingest_date': '2026-02-04', 'active_vehicles': '517', 'active_routes': '160', 'movement_points': '204141', 'avg_speed_kmh': '16.973134932693082', 'gold_author_surname': 'Yermolovych', 'schema_v': '1', 'batch_id': '42104c1c-5208-4fd9-9cdb-7e3675cd951d'}
{'ingest_date': '2026-02-05', 'active_vehicles': '506', 'active_routes': '153', 'movement_points': '175831', 'avg_speed_kmh': '15.157986418219057', 'gold_author_surname': 

## 6. Summary

In [8]:
summary = {
    "rows_before_cleaning": rows_before,
    "rows_after_cleaning": rows_after,
    "rows_rejected": rows_rejected,
    "written_metric_paths": written_paths,
}

print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "rows_before_cleaning": 9009537,
  "rows_after_cleaning": 9009537,
  "rows_rejected": 0,
  "written_metric_paths": [
    "E:\\Academy of Mohyla\\Course3_Stage2\\bigData\\data\\processed\\gold\\metrics\\traffic_activity_daily\\schema_v=1\\batch_id=42104c1c-5208-4fd9-9cdb-7e3675cd951d",
    "E:\\Academy of Mohyla\\Course3_Stage2\\bigData\\data\\processed\\gold\\metrics\\avg_speed_by_period\\schema_v=1\\batch_id=812f0f2e-c3ef-4647-a06c-23fe5f5c288b",
    "E:\\Academy of Mohyla\\Course3_Stage2\\bigData\\data\\processed\\gold\\metrics\\congestion_weather_daily\\schema_v=1\\batch_id=558f2aaf-fa46-44a7-adc1-92bdc5467334",
    "E:\\Academy of Mohyla\\Course3_Stage2\\bigData\\data\\processed\\gold\\metrics\\route_performance_daily\\schema_v=1\\batch_id=aa3ec766-b6b6-44bc-94ef-18bd6268c4b6",
    "E:\\Academy of Mohyla\\Course3_Stage2\\bigData\\data\\processed\\gold\\metrics\\geo_hotspots_daily\\schema_v=1\\batch_id=2934aac3-3906-4b43-9db9-b0d509369e86"
  ]
}
